# Tabular Classification Project: Trees

Build a classification pipeline using decision trees, random forests, and gradient boosting. This project practices everything from L07 — the progression from a single tree to an ensemble, feature importance, and model interpretation.

If you've also done the MLP project on the same dataset, compare results at the end. Trees and neural nets have different strengths on tabular data — seeing them side by side on the same problem is the best way to build intuition for when to use which.

In [ ]:
# Colab setup - run this cell first if you're on Google Colab
try:
    import google.colab
    !pip install -q xgboost lightgbm
    print("Colab environment ready!")
except ImportError:
    pass  # Not on Colab, no action needed

## Dataset Options

These are the same datasets available in the MLP project. Using the same one for both lets you compare approaches directly.

| Dataset | Rows | Cat:Num | Imbalance | Difficulty |
|---------|------|---------|-----------|------------|
| **Bank Marketing** | 41,188 | 10:10 | 88/12 | Medium |
| **Adult Income** | 48,842 | 8:6 | 76/24 | Medium |
| **Census-Income KDD** | 299,285 | ~24:16 | 94/6 | Hard |
| **Covertype** | 581,012 | 1:54 | Multi-class (7) | Hard |
| **Credit Card Fraud** | 284,807 | 0:30 | 99.8/0.2 | Hard |

### 1. Bank Marketing

Predict whether a client subscribes to a term deposit after a phone campaign. 41K rows, good mix of categorical and numeric features.

| Pros | Cons |
|------|------|
| Interpretable domain — feature importance tells a real story | `duration` feature leaks info (drop it) |
| Trees handle the categoricals natively with label encoding | 88/12 imbalance needs evaluation beyond accuracy |
| Good size for fast iteration | |

**Where:** Download from UCI (see `data/DATASETS.md`). Semicolon-delimited CSV.

**Recommended as first pick.** Interpretable features make SHAP plots meaningful.

### 2. Adult Income

Predict whether income exceeds $50K from census data. 48K rows, well-benchmarked.

| Pros | Cons |
|------|------|
| Tons of reference implementations to compare against | Older data (1994 census) |
| High-cardinality categoricals (occupation, country) | Missing values marked with `?` |
| 76/24 split — moderate imbalance | Overused in tutorials |

**Where:** `../../data/adult_income/adult.data` (download, see `data/DATASETS.md`)

**Safe pick.** Easy to sanity-check results against published benchmarks.

### 3. Census-Income KDD

Same domain as Adult Income but 6x larger with more features. 300K rows.

| Pros | Cons |
|------|------|
| Large enough to see boosting shine | No header row — need to parse column names from `.names` file |
| 94/6 imbalance forces serious evaluation | Lots of "Not in universe" values |

**Where:** Download from UCI (see `data/DATASETS.md`)

**For a challenge.** The preprocessing alone teaches real data engineering.

### 4. Covertype

Predict forest cover type from cartographic variables. 581K rows, 7 classes.

| Pros | Cons |
|------|------|
| Multi-class (7 types) — different from binary classification | Mostly numeric features, few categoricals |
| Very large — fast boosting implementations matter here | Some classes are rare |
| Available via `sklearn.datasets.fetch_covtype()` | |

**Best for** practicing multi-class metrics (per-class F1, confusion matrix) and seeing LightGBM's speed advantage over XGBoost at scale.

### 5. Credit Card Fraud

Detect fraudulent transactions. 284K rows, 99.8% legitimate.

| Pros | Cons |
|------|------|
| Extreme imbalance — forces you to think beyond accuracy | Features are PCA-transformed (anonymized), can't interpret them |
| Precision/recall tradeoff is very real here | No categoricals |
| Large, fast to load (single CSV) | SHAP plots are less meaningful on anonymous features |

**Where:** Kaggle download (see `data/DATASETS.md`)

**Best for** learning about extreme imbalance, precision/recall tradeoffs, and threshold tuning. Skip if you want interpretable feature importance.

## The Phases

```
1. Explore & prepare     →  Load, clean, split
2. Baseline tree         →  Single decision tree, visualize it
3. Random forest         →  Ensemble of trees, compare to baseline
4. Gradient boosting     →  XGBoost or LightGBM with early stopping
5. Feature importance    →  What drives predictions? SHAP if time allows
6. Evaluate & compare    →  Metrics, confusion matrix, compare all models
```

## Phase 1: Explore & Prepare

Load the data, handle missing values, encode categoricals (label encoding is fine for trees — no one-hot needed), and split train/validation/test. Stratify the split if classes are imbalanced.

Trees don't need feature scaling, but log-transforming a skewed target (if doing regression) helps RMSE.

## Phase 2: Baseline — Single Decision Tree

Train a single `DecisionTreeClassifier` with limited depth (e.g. `max_depth=5`). Visualize it with `sklearn.tree.plot_tree`. This is your interpretability baseline — you can literally read the rules.

Try an unrestricted tree too (`max_depth=None`). Compare train vs validation accuracy — the gap shows overfitting.

## Phase 3: Random Forest

Train a `RandomForestClassifier`. Start with 100 trees. Compare to the single tree — the ensemble should close the train/val gap while improving accuracy.

Things to try:
- `n_estimators`: 100 vs 300 vs 500 — diminishing returns?
- `max_features='sqrt'` vs `'log2'` — how much randomness per tree?
- `min_samples_leaf`: higher values = simpler trees = less overfitting

## Phase 4: Gradient Boosting

Train XGBoost or LightGBM with early stopping. Set `n_estimators` high (1000+) and let early stopping decide when to stop.

```python
model.fit(X_train, y_train,
          eval_set=[(X_val, y_val)],
          early_stopping_rounds=50,
          verbose=50)
```

If you have time, try both XGBoost and LightGBM and compare training speed and accuracy.

## Phase 5: Feature Importance

Plot feature importances from your best model. Which features matter most? Does the ranking match your intuition from Phase 1?

**Optional:** Use SHAP (`shap.TreeExplainer`) for richer interpretation. SHAP shows not just *which* features matter, but *how* — does high income push predictions up or down? The summary plot and waterfall plot are the most useful.

## Phase 6: Evaluate & Compare

Compare all models (single tree, random forest, gradient boosting) on the test set. Use metrics that matter for your class balance:

- **Balanced data:** accuracy, F1
- **Imbalanced data:** precision, recall, F1 on the minority class, ROC-AUC, confusion matrix

Build a summary table. If you also did the MLP project on the same dataset, add those results too.

## Reflection

1. How much did the random forest improve over a single tree? Was the improvement worth 100x the training time?
2. Did gradient boosting beat the random forest? By how much?
3. Which features mattered most? Any surprises?
4. If you also built an MLP on this data — which approach won, and why do you think that is?

<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
© 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>